<a href="https://colab.research.google.com/github/drfperez/openair/blob/main/XEMA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
import time
import urllib.parse
from google.colab import files
from datetime import datetime
import numpy as np

def descarregar_meteocat(any_inici=None, any_fi=None, codi_estacio=None, codi_variables=None):
    """
    Descarrega dades de la XEMA (Meteocat).
    Endpoint: https://analisi.transparenciacatalunya.cat/resource/nzvn-apee.json
    """
    base_url = "https://analisi.transparenciacatalunya.cat/resource/nzvn-apee.json"
    limit = 50000
    offset = 0
    tots_els_dades = []

    # Construir condicions SoQL
    condicions = []

    # Filtre per Estació
    if codi_estacio:
        condicions.append(f"codi_estacio='{codi_estacio}'")

    # Filtre per Variables (llista)
    if codi_variables:
        if len(codi_variables) > 1:
            in_clause = ','.join(f"'{v}'" for v in codi_variables)
            condicions.append(f"codi_variable IN ({in_clause})")
        else:
            condicions.append(f"codi_variable='{codi_variables[0]}'")

    # Filtre per Data (Rang d'anys)
    if any_inici:
        if any_fi is None:
            any_fi = any_inici
        data_inici = f"{any_inici}-01-01T00:00:00"
        data_fi = f"{any_fi + 1}-01-01T00:00:00"
        condicions.append(f"data_lectura >= '{data_inici}' AND data_lectura < '{data_fi}'")

    where_clause = " AND ".join(condicions) if condicions else "1=1"

    print("🚀 Iniciant descàrrega de dades METEOCAT (XEMA)...")
    print(f"   Filtres -> Anys: {any_inici} a {any_fi}, Estació: {codi_estacio}, Variables: {codi_variables}")

    while True:
        # Ordenem per data_lectura
        query = f"?$where={where_clause}&$limit={limit}&$offset={offset}&$order=data_lectura ASC"
        url_neta = base_url + urllib.parse.quote(query, safe='?=&-:T')

        try:
            df_bloc = pd.read_json(url_neta)

            if df_bloc.empty:
                break

            tots_els_dades.append(df_bloc)
            offset += limit

            # Mostrem progrés
            if 'data_lectura' in df_bloc.columns:
                ultima_data = df_bloc['data_lectura'].iloc[-1]
                print(f"📥 {offset} registres processats... Data actual: {str(ultima_data)[:10]}")
            else:
                print(f"📥 {offset} registres processats...")

            time.sleep(0.3)

        except Exception as e:
            print(f"⚠️ Error durant la descàrrega: {e}")
            break

    if tots_els_dades:
        df_final = pd.concat(tots_els_dades, ignore_index=True)

        # Convertim columnes de data
        if 'data_lectura' in df_final.columns:
            df_final['data_lectura'] = pd.to_datetime(df_final['data_lectura'])

        print(f"\n✅ ÈXIT: {len(df_final)} registres descarregats.")
        if not df_final.empty and 'data_lectura' in df_final.columns:
            print(f"📅 Període: {df_final['data_lectura'].min().date()} fins a {df_final['data_lectura'].max().date()}")

        # Convertir valor_lectura a numèric
        df_final['valor_lectura'] = pd.to_numeric(df_final['valor_lectura'], errors='coerce')

        # Seleccionar columnes rellevants
        df_final = df_final[['data_lectura', 'codi_variable', 'valor_lectura']]

        # Pivotar a format wide, amb aggfunc='mean' per manejar duplicats
        df_pivot = pd.pivot_table(df_final, index='data_lectura', columns='codi_variable', values='valor_lectura', aggfunc='mean')

        # Ordenar l'índex
        df_pivot = df_pivot.sort_index()

        # Definir mètodes d'agregació
        agg_dict = {}
        for col in df_pivot.columns:
            if col in ['35', '1400', '1401']:  # Precipitació i irradiacions diàries: suma
                agg_dict[col] = 'sum'
            else:  # Altres: mitjana
                agg_dict[col] = 'mean'

        # Resamplejar a horari
        df_hourly = df_pivot.resample('h').agg(agg_dict)

        # Reset index i renombrar
        df_hourly.reset_index(inplace=True)
        df_hourly.rename(columns={'data_lectura': 'date'}, inplace=True)

        # Renombrar columnes de variables a noms estàndard per openair
        var_names = {
            '30': 'ws',     # Velocitat vent
            '31': 'wd',     # Direcció vent
            '32': 'temp',   # Temperatura
            '33': 'rh',     # Humitat relativa
            '34': 'pres',   # Pressió
            '35': 'precip', # Precipitació
            '36': 'solar',  # Irradiància solar global
            # Afegeix més si cal, ex: '59': 'net_rad'
        }
        df_hourly.rename(columns=var_names, inplace=True)

        # Nom de fitxer dinàmic
        nom_fitxer = f"Meteocat_XEMA"
        if codi_estacio: nom_fitxer += f"_{codi_estacio}"
        if codi_variables: nom_fitxer += f"_Vars{'_'.join(codi_variables)}"
        if any_inici: nom_fitxer += f"_{any_inici}"
        if any_fi and any_fi != any_inici: nom_fitxer += f"-{any_fi}"
        nom_fitxer += "_openair_hourly.csv"

        df_hourly.to_csv(nom_fitxer, index=False, date_format='%Y-%m-%d %H:%M:%S')
        files.download(nom_fitxer)

        return df_hourly
    else:
        print("❌ No s'han trobat dades amb aquests filtres.")
        return None

# --- INTERACCIÓ AMB L'USUARI ---

print("--- DESCARREGADOR DADES METEOCAT (XEMA) ---")
print("ℹ️  Guia ràpida de codis:")
print("   VARIABLES COMUNES: 32 (Temp), 33 (Humitat), 35 (Pluja), 30 (Vent Vel.), 31 (Vent Dir.),")
print("                      34 (Pres), 36 (Irrad. Solar)")
print("   Nota: Aquest dataset és per dades meteorològiques. Per contaminants com O3, NO2, etc.,")
print("         consulta el dataset d'immissió de contaminants (XVPCA) en un endpoint diferent.")
print("   EXEMPLES ESTACIONS: X4 (Bcn-Raval), D5 (Bcn-Zoo), WU (Badalona), Y9 (Girona), XC (Castellbisbal)")
print("----------------------------------------------------")

any_inici_usuari = input("Introdueix l'any inici (deixa buit per tots): ").strip()
any_fi_usuari = input("Introdueix l'any fi (deixa buit si igual a inici): ").strip()
estacio_usuari = input("Introdueix Codi Estació (ex: X4, D5...): ").strip().upper()
variables_usuari = input("Introdueix Codis Variables separats per comes (ex: 32,33,35...): ").strip()

# Processar inputs
any_inici_usuari = int(any_inici_usuari) if any_inici_usuari.isdigit() else None
any_fi_usuari = int(any_fi_usuari) if any_fi_usuari.isdigit() else None
estacio_usuari = estacio_usuari if estacio_usuari else None
if variables_usuari:
    codi_variables = [v.strip() for v in variables_usuari.split(',')]
else:
    codi_variables = None

# Executar funció
df_resultat = descarregar_meteocat(any_inici_usuari, any_fi_usuari, estacio_usuari, codi_variables)

--- DESCARREGADOR DADES METEOCAT (XEMA) ---
ℹ️  Guia ràpida de codis:
   VARIABLES COMUNES: 32 (Temp), 33 (Humitat), 35 (Pluja), 30 (Vent Vel.), 31 (Vent Dir.),
                      34 (Pres), 36 (Irrad. Solar)
   Nota: Aquest dataset és per dades meteorològiques. Per contaminants com O3, NO2, etc.,
         consulta el dataset d'immissió de contaminants (XVPCA) en un endpoint diferent.
   EXEMPLES ESTACIONS: X4 (Bcn-Raval), D5 (Bcn-Zoo), WU (Badalona), Y9 (Girona), XC (Castellbisbal)
----------------------------------------------------
Introdueix l'any inici (deixa buit per tots): 2025
Introdueix l'any fi (deixa buit si igual a inici): 
Introdueix Codi Estació (ex: X4, D5...): X4
Introdueix Codis Variables separats per comes (ex: 32,33,35...): 30,31,32,33,34,35,36
🚀 Iniciant descàrrega de dades METEOCAT (XEMA)...
   Filtres -> Anys: 2025 a 2025, Estació: X4, Variables: ['30', '31', '32', '33', '34', '35', '36']
📥 50000 registres processats... Data actual: 2025-05-30
📥 100000 registre

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:

import pandas as pd

# Mostrar complet
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 0)

# URLs de metadades
URL_ESTACIONS = "https://analisi.transparenciacatalunya.cat/api/views/yqwd-vj5e/rows.csv?accessType=DOWNLOAD"
URL_VARIABLES = "https://analisi.transparenciacatalunya.cat/api/views/4fb2-n3yi/rows.csv?accessType=DOWNLOAD"

# --- ESTACIONS ---
print("⬇️ Descarregant metadades d’estacions XEMA...")
df_estacions = pd.read_csv(URL_ESTACIONS)

print("\nColumnes disponibles al CSV d'estacions:")
print(df_estacions.columns.tolist())

# Buscar columnes de codi i nom
codi_est_col = [c for c in df_estacions.columns if "codi" in c.lower()][0]
nom_est_col = [c for c in df_estacions.columns if "nom" in c.lower()][0]

df_estacions_neta = df_estacions[[codi_est_col, nom_est_col]].drop_duplicates().sort_values(codi_est_col)
df_estacions_neta.columns = ["codi_estacio", "nom_estacio"]

print("\n=== ESTACIONS XEMA (codi + nom) ===")
display(df_estacions_neta)
df_estacions_neta.to_csv("estacions_XEMA_cod_nom.csv", index=False)

# --- VARIABLES ---
print("\n⬇️ Descarregant metadades de variables XEMA...")
df_variables = pd.read_csv(URL_VARIABLES)

print("\nColumnes disponibles al CSV de variables:")
print(df_variables.columns.tolist())

# Buscar columnes de codi i nom
codi_var_col = [c for c in df_variables.columns if "codi" in c.lower()][0]
nom_var_col = [c for c in df_variables.columns if "nom" in c.lower()][0]

df_variables_neta = df_variables[[codi_var_col, nom_var_col]].drop_duplicates().sort_values(codi_var_col)
df_variables_neta.columns = ["codi_variable", "nom_variable"]

print("\n=== VARIABLES XEMA (codi + nom) ===")
display(df_variables_neta)
df_variables_neta.to_csv("variables_XEMA_cod_nom.csv", index=False)

print("\n✔ CSV generats:")
print(" - estacions_XEMA_cod_nom.csv")
print(" - variables_XEMA_cod_nom.csv")

⬇️ Descarregant metadades d’estacions XEMA...

Columnes disponibles al CSV d'estacions:
['CODI_ESTACIO', 'NOM_ESTACIO', 'CODI_TIPUS', 'LATITUD', 'LONGITUD', 'Georeferència', 'EMPLACAMENT', 'ALTITUD', 'CODI_MUNICIPI', 'NOM_MUNICIPI', 'CODI_COMARCA', 'NOM_COMARCA', 'CODI_PROVINCIA', 'NOM_PROVINCIA', 'CODI_XARXA', 'NOM_XARXA', 'CODI_ESTAT', 'NOM_ESTAT', 'DATA_ALTA', 'DATA_BAIXA']

=== ESTACIONS XEMA (codi + nom) ===


,codi_estacio,nom_estacio
161,AN,Barcelona - Av. Lluís Companys
42,C6,Castellnou de Seana
39,C7,Tàrrega
244,C8,Cervera
231,C9,Mas de Barberans
227,CA,Clariana de Cardener
118,CB,les Llosses
99,CC,Orís
116,CD,la Seu d'Urgell - Bellestar
211,CE,els Hostalets de Pierola



⬇️ Descarregant metadades de variables XEMA...

Columnes disponibles al CSV de variables:
['CODI_VARIABLE', 'NOM_VARIABLE', 'UNITAT', 'ACRONIM', 'CODI_TIPUS_VAR', 'DECIMALS']

=== VARIABLES XEMA (codi + nom) ===


,codi_variable,nom_variable
10,1,Pressió atmosfèrica màxima
12,2,Pressió atmosfèrica mínima
6,3,Humitat relativa màxima
7,30,Velocitat del vent a 10 m (esc.)
9,31,Direcció de vent 10 m (m. 1)
3,32,Temperatura
23,33,Humitat relativa
16,34,Pressió atmosfèrica
19,35,Precipitació
13,36,Irradiància solar global



✔ CSV generats:
 - estacions_XEMA_cod_nom.csv
 - variables_XEMA_cod_nom.csv
